Until now no luck in finding a valid digit to be an exemplary digit for a network. All the constructions seem to be random noise. All of them were similar though.

There are still 2 more tricks remaining. The first is to try another ensemble but with other types of networks other than fully connected. The second method is to use data augmentation and an 11th output for the network representing not-a-digit, then training the network to recognize digits and noise (not digits).

In this notebook the first method is used.

In [ ]:
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch import no_grad
from torch import argmax
from torchvision import transforms
from torch import cuda
from torch import save
from torch import zeros

In [ ]:
# first a fully connected 784,784,784,10
class Ann(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(784, 784)
    self.fc2 = nn.Linear(784, 784)
    self.fc3 = nn.Linear(784, 10)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = x.view(x.shape[0], -1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.fc3(x)
    return x

# now a cnn
class Cnn(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 16, 3)
    self.conv2 = nn.Conv2d(16, 32, 3)
    self.conv3 = nn.Conv2d(32, 64, 3)
    # max pooling
    self.pool = nn.MaxPool2d(2, 2)
    # fully connected
    self.fc1 = nn.Linear(64, 10) # the shape becomes (64, 1, 1) after the transformations

  def forward(self, x):
    x = self.conv1(x)
    x = self.pool(x)
    x = self.conv2(x)
    x = self.pool(x)
    x = self.conv3(x)
    x = self.pool(x)
    x = x.view(x.shape[0], -1)
    x = self.fc1(x)
    return x

# a bidirectional rnn
class Bidirectional_rnn(nn.Module):
  def __init__(self):
    super().__init__()
    self.rnn = nn.RNN(28, 128, 2, batch_first=True, bidirectional=True)
    self.fc = nn.Linear(256, 10)

  def forward(self, x):
    x = x.squeeze(dim=1)
    x, _ = self.rnn(x)
    x = x[:, -1, :]
    x = self.fc(x)
    return x

# a gru
class Gru_rows(nn.Module):
  def __init__(self):
    super().__init__()
    self.gru = nn.GRU(28, 128, 2, batch_first=True, bidirectional=True)
    self.fc = nn.Linear(256, 10)

  def forward(self, x):
    x = x.squeeze(dim=1)
    x, _ = self.gru(x)
    x = x[:, -1, :]
    x = self.fc(x)
    return x

# gru but passes columns as sequences instead of rows
class Gru_columns(nn.Module):
  def __init__(self):
    super().__init__()
    self.gru = nn.GRU(28, 128, 2, batch_first=True, bidirectional=True)
    self.fc = nn.Linear(256, 10)

  def forward(self, x):
    x = x.squeeze(dim=1)
    x = x.permute(0, 2, 1) # switch rows and columns, leave batch unchanged
    x, _ = self.gru(x)
    x = x[:, -1, :]
    x = self.fc(x)
    return x


In [ ]:
# initialize the dataloader for mnist
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False)

In [ ]:
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
# hyperparameters
lr = 0.001
epochs = 10

In [ ]:
# create the models
judges = [Ann(), Cnn(), Bidirectional_rnn(), Gru_rows(), Gru_columns()]

In [ ]:
for network in judges:
  network.to(device)

In [ ]:
# train all of them
for i, network in enumerate(judges):
  optimizer = Adam(network.parameters(), lr=lr)
  criterion = nn.CrossEntropyLoss()

  for epoch in range(epochs):
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      output = network(images)
      loss = criterion(output, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    # calculate accuracy on validation set
    total = 0
    correct = 0
    with no_grad():
      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        output = network(images)
        predictions = argmax(output, dim=1)
        total += labels.shape[0]
        correct += (predictions == labels).sum().item()
      print(f'Judge number: {i+1}. Epoch: {epoch+1}. Train Loss: {loss.item()}. Test Accuracy: {100*correct/total:.2f}')

Judge number: 1. Epoch: 1. Train Loss: 0.22231470048427582. Test Accuracy: 97.49
Judge number: 1. Epoch: 2. Train Loss: 0.11667784303426743. Test Accuracy: 97.61
Judge number: 1. Epoch: 3. Train Loss: 0.21173632144927979. Test Accuracy: 97.69
Judge number: 1. Epoch: 4. Train Loss: 0.018033437430858612. Test Accuracy: 98.13
Judge number: 1. Epoch: 5. Train Loss: 0.00031240424141287804. Test Accuracy: 98.04
Judge number: 1. Epoch: 6. Train Loss: 0.04161927103996277. Test Accuracy: 98.21
Judge number: 1. Epoch: 7. Train Loss: 0.0004941451479680836. Test Accuracy: 98.04
Judge number: 1. Epoch: 8. Train Loss: 0.0008934361394494772. Test Accuracy: 98.17
Judge number: 1. Epoch: 9. Train Loss: 0.0008314745500683784. Test Accuracy: 97.60
Judge number: 1. Epoch: 10. Train Loss: 0.00036113231908529997. Test Accuracy: 97.61
Judge number: 2. Epoch: 1. Train Loss: 0.06268874555826187. Test Accuracy: 98.12
Judge number: 2. Epoch: 2. Train Loss: 0.008882343769073486. Test Accuracy: 97.93
Judge number:

In [ ]:
# create folder ensemble
!mkdir rnn_cnn_ensemble
# saving the models
for i, network in enumerate(judges):
  save(network.state_dict(), f'rnn_cnn_ensemble/judge_{i+1}.pth')

In [ ]:
# zip it
!zip -r rnn_cnn_ensemble.zip rnn_cnn_ensemble

  adding: rnn_cnn_ensemble/ (stored 0%)
  adding: rnn_cnn_ensemble/judge_3.pth (deflated 8%)
  adding: rnn_cnn_ensemble/judge_1.pth (deflated 7%)
  adding: rnn_cnn_ensemble/judge_4.pth (deflated 7%)
  adding: rnn_cnn_ensemble/judge_2.pth (deflated 8%)
  adding: rnn_cnn_ensemble/judge_5.pth (deflated 7%)
